# Day 22: A/B Test Analysis

**Dataset:** `AB_Testing_Sample_Dataset.csv`

This notebook analyzes an A/B test comparing a `control` group (shown the old landing page)
against a `treatment` group (shown a new landing page), and determines whether the observed
differences in conversion rate (and a secondary metric, purchase amount) are statistically
meaningful, using real values pulled directly from the dataset.

## 1. Import Libraries and Load the Dataset

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the dataset
df = pd.read_csv("../datasets/AB_Testing_Sample_Dataset.csv")

# Preview the data
df.head()

,user_id,timestamp,group,landing_page,converted,age,gender,location,session_duration,pages_visited,device_type,purchase_amount
0,U1,2025-08-02 15:27:54.137058,control,old_page,0,37,Male,Pakistan,3.69,4,Mobile,0.0
1,U2,2024-04-22 10:22:51.712050,treatment,new_page,0,31,Female,UK,1.29,3,Desktop,0.0
2,U3,2024-08-14 21:35:11.135894,treatment,new_page,0,38,Male,US,3.72,5,Desktop,0.0
3,U4,2025-03-19 03:28:51.120807,treatment,new_page,0,28,Female,India,7.76,2,Mobile,0.0
4,U5,2024-12-22 13:13:17.973162,control,old_page,0,33,Male,Australia,6.78,6,Mobile,0.0


In [2]:
print(df.shape)
df.info()

(294478, 12)
<class 'pandas.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   user_id           294478 non-null  str    
 1   timestamp         294478 non-null  str    
 2   group             294478 non-null  str    
 3   landing_page      294478 non-null  str    
 4   converted         294478 non-null  int64  
 5   age               294478 non-null  int64  
 6   gender            294478 non-null  str    
 7   location          294478 non-null  str    
 8   session_duration  294478 non-null  float64
 9   pages_visited     294478 non-null  int64  
 10  device_type       294478 non-null  str    
 11  purchase_amount   294478 non-null  float64
dtypes: float64(2), int64(3), str(7)
memory usage: 27.0 MB


## 2. Defining the Control and Treatment Groups

- **Control group:** users who saw the `old_page` (the existing landing page).
- **Treatment group:** users who saw the `new_page` (the new landing page being tested).

The primary metric being tested is `converted` (1 = the user converted, 0 = they didn't). A
secondary metric, `purchase_amount`, is also compared.

In [3]:
print(df["group"].value_counts())
print()
print("group vs landing_page (confirming each group only saw its own page):")
print(pd.crosstab(df["group"], df["landing_page"]))

group
treatment    147552
control      146926
Name: count, dtype: int64

group vs landing_page (confirming each group only saw its own page):
landing_page  new_page  old_page
group                           
control              0    146926
treatment       147552         0


In [4]:
control = df[df["group"] == "control"]
treatment = df[df["group"] == "treatment"]

print("Control group size:", len(control))
print("Treatment group size:", len(treatment))

Control group size: 146926
Treatment group size: 147552


## 3. Descriptive Statistics for Both Groups

**Conversion rate for each group**

In [5]:
conversion_summary = df.groupby("group")["converted"].agg(
    users="count", conversions="sum", conversion_rate="mean"
)
conversion_summary

,users,conversions,conversion_rate
group,,,
control,146926,17444,0.118726
treatment,147552,26484,0.179489


**Purchase amount and session duration for each group**

In [6]:
df.groupby("group")[["purchase_amount", "session_duration", "pages_visited"]].agg(["mean", "std"])

purchase_amount            session_duration           pages_visited  \
                     mean        std             mean       std          mean   
group                                                                           
control          4.452769  13.924962         5.003684  1.979324      4.015327   
treatment        6.763299  16.730481         5.000594  1.981246      4.023754   

                     
                std  
group                
control    1.965816  
treatment  1.964548

## 4. Statistical Test on the Primary Metric: Conversion Rate

Since `converted` is a binary outcome, a **two-proportion z-test** is the appropriate test for
comparing the control and treatment conversion rates.

In [7]:
n_control, x_control = len(control), control["converted"].sum()
n_treatment, x_treatment = len(treatment), treatment["converted"].sum()

p_control = x_control / n_control
p_treatment = x_treatment / n_treatment

# Pooled proportion, under the null hypothesis that both groups have the same true rate
p_pooled = (x_control + x_treatment) / (n_control + n_treatment)
se_pooled = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n_control + 1 / n_treatment))

z_stat = (p_treatment - p_control) / se_pooled
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print(f"Control conversion rate:   {p_control:.4f}")
print(f"Treatment conversion rate: {p_treatment:.4f}")
print(f"Z-statistic: {z_stat:.2f}")
print(f"P-value: {p_value}")

Control conversion rate:   0.1187
Treatment conversion rate: 0.1795
Z-statistic: 46.28
P-value: 0.0


**Effect size: absolute difference, relative lift, and a 95% confidence interval**

In [8]:
absolute_diff = p_treatment - p_control
relative_lift = (absolute_diff / p_control) * 100

# Confidence interval uses the UNPOOLED standard error (standard for a CI on the difference)
se_diff = np.sqrt(p_control * (1 - p_control) / n_control + p_treatment * (1 - p_treatment) / n_treatment)
ci_lower = absolute_diff - 1.96 * se_diff
ci_upper = absolute_diff + 1.96 * se_diff

print(f"Absolute difference in conversion rate: {absolute_diff:.4f} ({absolute_diff*100:.2f} percentage points)")
print(f"Relative lift: {relative_lift:.1f}%")
print(f"95% confidence interval for the difference: ({ci_lower:.4f}, {ci_upper:.4f})")

Absolute difference in conversion rate: 0.0608 (6.08 percentage points)
Relative lift: 51.2%
95% confidence interval for the difference: (0.0582, 0.0633)


## 5. Statistical Test on a Secondary Metric: Purchase Amount

For a continuous metric like `purchase_amount`, an independent-samples **t-test** (Welch's
version, which doesn't assume equal variances) is appropriate.

In [9]:
t_stat, t_pvalue = stats.ttest_ind(
    treatment["purchase_amount"], control["purchase_amount"], equal_var=False
)

print(f"Control mean purchase amount:   {control['purchase_amount'].mean():.2f}")
print(f"Treatment mean purchase amount: {treatment['purchase_amount'].mean():.2f}")
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {t_pvalue}")

Control mean purchase amount:   4.45
Treatment mean purchase amount: 6.76
T-statistic: 40.74
P-value: 0.0


**Effect size for the continuous metric: Cohen's d**

`std()` and the difference in means alone don't tell you how *large* the effect really is,
relative to how spread out the data is — Cohen's d standardizes that difference.

In [10]:
pooled_std = np.sqrt(
    ((len(control) - 1) * control["purchase_amount"].std() ** 2 +
     (len(treatment) - 1) * treatment["purchase_amount"].std() ** 2)
    / (len(control) + len(treatment) - 2)
)

cohens_d = (treatment["purchase_amount"].mean() - control["purchase_amount"].mean()) / pooled_std
print(f"Cohen's d: {cohens_d:.3f}")
print("(Cohen's convention: ~0.2 = small, ~0.5 = medium, ~0.8 = large effect)")

Cohen's d: 0.150
(Cohen's convention: ~0.2 = small, ~0.5 = medium, ~0.8 = large effect)


## 6. Why Significance Alone Isn't Enough

This dataset is a good example of exactly why you shouldn't decide based only on a p-value:

- **Conversion rate:** p-value is essentially 0 (highly "significant") AND the effect is
  large and practically meaningful (a ~51% relative lift in conversions).
- **Purchase amount:** p-value is also essentially 0 (highly "significant"), but Cohen's d is
  only about **0.15**, which by Cohen's own convention counts as a **small** effect. With a
  sample size this large (nearly 300,000 users), even a tiny, possibly not-very-meaningful
  difference in average purchase amount will come out "statistically significant." The p-value
  alone would make both results look equally convincing since only the effect size reveals that
  one is a strong result and the other is a much weaker one.

## 7. Final Recommendation Based on Evidence

**Recommendation: launch the new page.**

- The conversion rate improved from **11.87%** (control) to **17.95%** (treatment), an
  absolute lift of about **6.1 percentage points** and a **~51% relative lift**, with a 95%
  confidence interval of roughly **(5.8, 6.3) percentage points** that does not include zero.
  This is both statistically significant (z ≈ 46, p ≈ 0) and a large, practically meaningful
  effect — exactly the kind of result an A/B test is designed to detect.
- The secondary metric (`purchase_amount`) also increased in the treatment group and was
  statistically significant, but with a small effect size (Cohen's d ≈ 0.15). This is
  supporting evidence, not the main basis for the decision, where the conversion rate result alone
  is strong enough to justify the change.
- Because both a large effect size and strong statistical significance point in the same
  direction, this is a case where the evidence is unambiguous, rather than one that hinges on
  a borderline p-value.

## Outcome

This notebook practiced defining control and treatment groups, computing descriptive
statistics for each, and running two appropriate statistical tests: a two-proportion z-test
for the binary `converted` metric and a Welch's t-test for the continuous `purchase_amount`
metric. Calculating effect size alongside each p-value (absolute/relative lift and a
confidence interval for the conversion rate, and Cohen's d for purchase amount) showed
directly why a p-value by itself isn't enough to make a decision: both metrics came back
highly "significant," but only the conversion rate result also had a large, practically
meaningful effect size, while the purchase amount effect was statistically real but small.
Combining significance and effect size together led to a confident, evidence-based
recommendation to launch the new landing page.